# Part II · Lesson 3 — Put an LLM inside the graph

The model call stays familiar; the graph controls where it belongs.

Run cells from the top in the code project environment.

## Set up once

Use the same provider connection from Part I.

### Import the model and graph helpers

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END

**What happens here**

These are familiar LangChain model and message helpers, plus our graph building blocks. MessagesState already defines a messages list with the add_messages reducer, so we do not need to write that same schema again.

**Result:** The model, message, and graph helpers are available.

### Load the key and create the model

In [ ]:
# Keep GROQ_API_KEY in the local .env file.
load_dotenv()
model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
    reasoning_effort="low",
)

**What happens here**

We use the same Groq model as Part I. load_dotenv makes the key available, and init_chat_model prepares our connection. No question has been sent yet.

**Result:** model is ready for the node that will call it.

## Define and connect the node

Notice how little the model call changes.

### Put the model call inside a node

In [ ]:
def chatbot(state: MessagesState) -> dict:
    reply = model.invoke([
        SystemMessage("You are a concise assistant for a Python learner."),
        *state["messages"],
    ])
    # Return only the new reply; the reducer keeps the earlier messages.
    return {"messages": [reply]}

**What happens here**

Our node receives the current messages, adds an instruction for this model call, and invokes the model. The star expands the existing list into the request. We return the new AIMessage itself, preserving its metadata and any structured fields. We do not return the whole old conversation again.

**Result:** The node adds one model reply to the graph state.

### Connect and compile the chat graph

In [ ]:
builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
graph = builder.compile()

**What happens here**

The flow has one working node. START sends the input to chatbot, and END finishes after its reply. This is intentionally small: adding a graph around one call helps us see where later steps will go.

**Result:** graph is ready to accept an initial messages list.

## Run a model-backed workflow

Send the user's message as graph state.

### Invoke the graph

In [ ]:
response = graph.invoke({
    "messages": [HumanMessage("Explain a REST API in two sentences.")]
})
print(response["messages"][-1].text)

**What happens here**

The graph passes the initial message to chatbot. Its returned AIMessage is merged into state. We read the last message for the answer instead of trying response.text, because the graph returns a state dictionary.

**Result:** response contains the human request and the model reply. The wording may vary.

### See what the graph returned

In [ ]:
for message in response["messages"]:
    print(message.type, ":", message.text)

**What happens here**

We inspect the conversation roles and text. The system instruction was supplied inside the node's model call; it was not appended to the stored messages list.

**Result:** The state contains one human message followed by one AI message.